# Análise de Agrupamento: K-means e PCA

**Projeto:** Análise de Dados de Compliance Público (TCC MBA)  
**Autor:** Enok  
**Última Atualização:** 2026-02-10

---

## Objetivo

Este notebook realiza análise de agrupamento em municípios brasileiros utilizando indicadores socioeconômicos dos Censos de 2010 e 2022. A análise inclui:

1. **Estatísticas Descritivas** - Resumo do dataset consolidado
2. **PCA (Análise de Componentes Principais)** - Redução de dimensionalidade para identificar componentes de variância
3. **Agrupamento K-means** - Agrupar municípios baseado em:
   - População
   - Taxas de alfabetização
   - Renda
   - Indicadores sociais de domicílios
4. **Perfil dos Clusters** - Caracterização de cada grupo

---

## Fonte de Dados

- **Dataset:** `gold/consolidated_clustering/data.parquet`
- **Granularidade:** Um registro por município (cidade)
- **Características:**
  - Sem municípios duplicados
  - Sem valores ausentes nas features de agrupamento
  - Features pré-normalizadas (padronização z-score)
  - Dados de ambos os anos censitários (2010 e 2022)

## 1. Configuração e Carregamento de Dados

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Adicionar raiz do projeto ao path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Bibliotecas principais
import numpy as np
import pandas as pd

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# AWS
import boto3
import tempfile

# Configurar estilo de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Bibliotecas carregadas com sucesso!")

In [ ]:
# Configuração
BUCKET_NAME = "enok-mba-thesis-datalake"
DATA_KEY = "gold/consolidated_clustering/data.parquet"

# Carregar dados do S3
def load_from_s3(bucket: str, key: str) -> pd.DataFrame:
    """Carregar arquivo parquet do S3."""
    aws_profile = os.getenv("AWS_PROFILE", "mba-thesis")
    session = boto3.Session(profile_name=aws_profile)
    s3 = session.client('s3')
    with tempfile.NamedTemporaryFile(suffix='.parquet') as tmp:
        s3.download_file(bucket, key, tmp.name)
        return pd.read_parquet(tmp.name)

# Carregar o dataset consolidado
df = load_from_s3(BUCKET_NAME, DATA_KEY)
print(f"Carregados {len(df):,} municípios")
print(f"Colunas: {len(df.columns)}")

In [ ]:
# Exibir primeiras linhas
df.head()

In [ ]:
# Verificar qualidade dos dados
print("=" * 60)
print("VERIFICAÇÃO DE QUALIDADE DOS DADOS")
print("=" * 60)
print(f"\nTotal de municípios: {len(df):,}")
print(f"Municípios únicos: {df['municipality_code'].nunique():,}")
print(f"Municípios duplicados: {len(df) - df['municipality_code'].nunique()}")
print(f"\nValores ausentes por coluna:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Sem valores ausentes!")

## 2. Estatísticas Descritivas

In [ ]:
# Definir colunas de features brutas (não normalizadas)
raw_features = [
    'population_2010', 'population_2022', 'population_change_pct',
    'literacy_rate_2010', 'literacy_rate_2022', 'literacy_change_pp',
    'avg_income_2010', 'avg_income_2022', 'income_change_pct',
    'households_2010', 'households_2022', 'households_change_pct'
]

# Estatísticas descritivas
print("=" * 80)
print("ESTATÍSTICAS DESCRITIVAS - Features Brutas")
print("=" * 80)
df[raw_features].describe().round(2).T

In [ ]:
# Distribuição por região
print("\n" + "=" * 60)
print("DISTRIBUIÇÃO POR REGIÃO")
print("=" * 60)

region_stats = df.groupby('region_name').agg({
    'municipality_code': 'count',
    'population_2022': ['sum', 'mean', 'median'],
    'literacy_rate_2022': 'mean',
    'avg_income_2022': 'mean'
}).round(2)

region_stats.columns = ['N_Municipios', 'Pop_Total', 'Pop_Media', 'Pop_Mediana', 
                        'Alfabetizacao_Media', 'Renda_Media']
region_stats = region_stats.sort_values('N_Municipios', ascending=False)
region_stats

In [ ]:
# Distribuição por estado
print("\n" + "=" * 60)
print("TOP 10 ESTADOS POR NÚMERO DE MUNICÍPIOS")
print("=" * 60)

state_counts = df.groupby(['state_name', 'region_name']).size().reset_index(name='n_municipios')
state_counts = state_counts.sort_values('n_municipios', ascending=False).head(10)
state_counts

In [ ]:
# Visualizar distribuições das features principais
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# População 2022 (escala log)
ax = axes[0, 0]
ax.hist(np.log10(df['population_2022']), bins=50, edgecolor='white', alpha=0.7)
ax.set_xlabel('Log10(População 2022)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de População (2022)')

# Taxa de Alfabetização 2022
ax = axes[0, 1]
ax.hist(df['literacy_rate_2022'], bins=50, edgecolor='white', alpha=0.7, color='green')
ax.set_xlabel('Taxa de Alfabetização (%)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição da Taxa de Alfabetização (2022)')

# Renda 2022
ax = axes[0, 2]
ax.hist(df['avg_income_2022'], bins=50, edgecolor='white', alpha=0.7, color='orange')
ax.set_xlabel('Renda Média (R$)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de Renda (2022)')

# Variação da População
ax = axes[1, 0]
ax.hist(df['population_change_pct'], bins=50, edgecolor='white', alpha=0.7, color='purple')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Variação da População (%)')
ax.set_ylabel('Frequência')
ax.set_title('Variação Populacional (2010-2022)')

# Variação da Alfabetização
ax = axes[1, 1]
ax.hist(df['literacy_change_pp'], bins=50, edgecolor='white', alpha=0.7, color='teal')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Variação da Alfabetização (pp)')
ax.set_ylabel('Frequência')
ax.set_title('Variação da Alfabetização (2010-2022)')

# Variação da Renda
ax = axes[1, 2]
ax.hist(df['income_change_pct'], bins=50, edgecolor='white', alpha=0.7, color='brown')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Variação da Renda (%)')
ax.set_ylabel('Frequência')
ax.set_title('Variação da Renda (2010-2022)')

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlação
plt.figure(figsize=(14, 10))
corr_matrix = df[raw_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Matriz de Correlação - Features Socioeconômicas', fontsize=14)
plt.tight_layout()
plt.show()

## 3. PCA - Análise de Componentes Principais

O PCA nos ajudará a:
1. Reduzir dimensionalidade preservando a variância
2. Identificar quais features mais contribuem para a variância
3. Visualizar municípios em espaço 2D/3D
4. Potencialmente usar menos componentes para agrupamento

In [ ]:
# Usar features normalizadas para PCA
norm_features = [f'{col}_norm' for col in raw_features]

# Extrair dados normalizados
X_norm = df[norm_features].values

print(f"Formato da matriz de features: {X_norm.shape}")
print(f"Número de features: {len(norm_features)}")

In [ ]:
# Executar PCA com todos os componentes
pca_full = PCA()
pca_full.fit(X_norm)

# Variância explicada
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Exibir variância explicada
print("=" * 60)
print("PCA - VARIÂNCIA EXPLICADA")
print("=" * 60)
pca_df = pd.DataFrame({
    'Componente': [f'PC{i+1}' for i in range(len(explained_var))],
    'Variancia_Explicada': explained_var * 100,
    'Variancia_Acumulada': cumulative_var * 100
})
print(pca_df.round(2).to_string(index=False))

In [ ]:
# Gráfico Scree e variância acumulada
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico Scree
ax = axes[0]
components = range(1, len(explained_var) + 1)
ax.bar(components, explained_var * 100, alpha=0.7, label='Individual')
ax.plot(components, cumulative_var * 100, 'ro-', label='Acumulada')
ax.axhline(y=80, color='green', linestyle='--', label='Limiar 80%')
ax.set_xlabel('Componente Principal')
ax.set_ylabel('Variância Explicada (%)')
ax.set_title('Gráfico Scree do PCA')
ax.legend()
ax.set_xticks(components)

# Variância acumulada
ax = axes[1]
ax.plot(components, cumulative_var * 100, 'b-o', linewidth=2, markersize=8)
ax.axhline(y=80, color='green', linestyle='--', label='Limiar 80%')
ax.axhline(y=90, color='orange', linestyle='--', label='Limiar 90%')
ax.axhline(y=95, color='red', linestyle='--', label='Limiar 95%')
ax.fill_between(components, cumulative_var * 100, alpha=0.3)
ax.set_xlabel('Número de Componentes')
ax.set_ylabel('Variância Acumulada Explicada (%)')
ax.set_title('Variância Acumulada Explicada')
ax.legend()
ax.set_xticks(components)

plt.tight_layout()
plt.show()

# Determinar componentes ótimos
n_components_80 = np.argmax(cumulative_var >= 0.80) + 1
n_components_90 = np.argmax(cumulative_var >= 0.90) + 1
print(f"\nComponentes necessários para 80% da variância: {n_components_80}")
print(f"Componentes necessários para 90% da variância: {n_components_90}")

In [ ]:
# Cargas do PCA (contribuições das features para cada componente)
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=[f'PC{i+1}' for i in range(len(explained_var))],
    index=[col.replace('_norm', '') for col in norm_features]
)

print("=" * 60)
print("CARGAS DO PCA (Contribuições das Features)")
print("=" * 60)
print(loadings[['PC1', 'PC2', 'PC3', 'PC4']].round(3))

In [ ]:
# Visualizar heatmap das cargas
plt.figure(figsize=(12, 8))
sns.heatmap(loadings[['PC1', 'PC2', 'PC3', 'PC4']], annot=True, cmap='RdBu_r', 
            center=0, fmt='.2f', linewidths=0.5)
plt.title('Cargas do PCA - Contribuições das Features para Componentes Principais', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Transformar dados para componentes principais
pca_3 = PCA(n_components=3)
X_pca = pca_3.fit_transform(X_norm)

# Adicionar componentes PCA ao dataframe
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]
df['PC3'] = X_pca[:, 2]

print(f"Transformação PCA concluída.")
print(f"Variância explicada por 3 componentes: {pca_3.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# Visualização 2D do PCA por região
fig = px.scatter(
    df, x='PC1', y='PC2',
    color='region_name',
    hover_data=['municipality_name', 'state_name', 'population_2022', 'avg_income_2022'],
    title='PCA: Municípios no Espaço 2D (por Região)',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)'}
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# Visualização 3D do PCA
fig = px.scatter_3d(
    df, x='PC1', y='PC2', z='PC3',
    color='region_name',
    hover_data=['municipality_name', 'state_name'],
    title='PCA: Municípios no Espaço 3D (por Região)',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)',
            'PC3': f'PC3 ({pca_3.explained_variance_ratio_[2]*100:.1f}%)'}
)
fig.update_layout(height=700)
fig.show()

## 4. Agrupamento K-means

Utilizaremos K-means para agrupar municípios baseado em indicadores socioeconômicos:
- População
- Taxas de alfabetização
- Renda
- Indicadores de domicílios

### 4.1 Determinar Número Ótimo de Clusters

In [ ]:
# Selecionar features para agrupamento (usando features normalizadas)
clustering_features = [
    'population_2022_norm',
    'literacy_rate_2022_norm',
    'avg_income_2022_norm',
    'households_2022_norm',
    'population_change_pct_norm',
    'literacy_change_pp_norm',
    'income_change_pct_norm',
    'households_change_pct_norm'
]

X_cluster = df[clustering_features].values
print(f"Matriz de features para agrupamento: {X_cluster.shape}")
print(f"Features utilizadas: {clustering_features}")

In [ ]:
# Método do Cotovelo e Análise de Silhueta
k_range = range(2, 11)
inertias = []
silhouettes = []

print("Avaliando valores de K...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster)
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_cluster, kmeans.labels_)
    silhouettes.append(sil_score)
    print(f"  K={k}: Inércia={kmeans.inertia_:.0f}, Silhueta={sil_score:.4f}")

In [ ]:
# Plotar Cotovelo e Silhueta
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico do Cotovelo
ax = axes[0]
ax.plot(list(k_range), inertias, 'b-o', linewidth=2, markersize=8)
ax.set_xlabel('Número de Clusters (K)')
ax.set_ylabel('Inércia (Soma dos quadrados intra-cluster)')
ax.set_title('Método do Cotovelo')
ax.set_xticks(list(k_range))

# Gráfico da Silhueta
ax = axes[1]
ax.plot(list(k_range), silhouettes, 'g-o', linewidth=2, markersize=8)
ax.set_xlabel('Número de Clusters (K)')
ax.set_ylabel('Coeficiente de Silhueta')
ax.set_title('Análise de Silhueta')
ax.set_xticks(list(k_range))

# Destacar melhor silhueta
best_k = list(k_range)[np.argmax(silhouettes)]
best_sil = max(silhouettes)
ax.axvline(x=best_k, color='red', linestyle='--', label=f'Melhor K={best_k}')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nMelhor K baseado no Coeficiente de Silhueta: {best_k} (score={best_sil:.4f})")

### 4.2 Aplicar K-means com K Ótimo

In [ ]:
# Usar K baseado na análise (ajustar conforme necessário)
OPTIMAL_K = best_k
print(f"Usando K = {OPTIMAL_K} clusters")

# Ajustar modelo K-means final
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_cluster)

# Distribuição dos clusters
print("\nDistribuição dos Clusters:")
print(df['cluster'].value_counts().sort_index())

In [ ]:
# Visualizar clusters no espaço PCA (2D)
fig = px.scatter(
    df, x='PC1', y='PC2',
    color='cluster',
    color_continuous_scale='viridis',
    hover_data=['municipality_name', 'state_name', 'region_name', 'population_2022', 'avg_income_2022'],
    title=f'Clusters K-means (K={OPTIMAL_K}) no Espaço PCA',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)',
            'cluster': 'Cluster'}
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(height=600)
fig.show()

In [ ]:
# Visualizar clusters no espaço PCA 3D
fig = px.scatter_3d(
    df, x='PC1', y='PC2', z='PC3',
    color='cluster',
    color_continuous_scale='viridis',
    hover_data=['municipality_name', 'state_name'],
    title=f'Clusters K-means (K={OPTIMAL_K}) no Espaço PCA 3D'
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(height=700)
fig.show()

### 4.3 Perfil dos Clusters

In [ ]:
# Estatísticas dos clusters
print("=" * 80)
print("PERFIS DOS CLUSTERS - Valores Médios")
print("=" * 80)

cluster_stats = df.groupby('cluster').agg({
    'municipality_code': 'count',
    'population_2022': ['mean', 'median'],
    'literacy_rate_2022': 'mean',
    'avg_income_2022': 'mean',
    'households_2022': 'mean',
    'population_change_pct': 'mean',
    'literacy_change_pp': 'mean',
    'income_change_pct': 'mean'
}).round(2)

cluster_stats.columns = ['N_Municipios', 'Pop_Media', 'Pop_Mediana', 'Alfabetizacao_Media',
                         'Renda_Media', 'Domicilios_Media', 'Var_Pop', 'Var_Alfab', 'Var_Renda']
cluster_stats

In [ ]:
# Composição dos clusters por região
print("\n" + "=" * 60)
print("COMPOSIÇÃO DOS CLUSTERS POR REGIÃO")
print("=" * 60)

region_cluster = pd.crosstab(df['cluster'], df['region_name'], margins=True)
print(region_cluster)

In [ ]:
# Visualizar composição dos clusters por região
fig = px.histogram(
    df, x='cluster', color='region_name',
    barmode='stack',
    title='Composição dos Clusters por Região',
    labels={'cluster': 'Cluster', 'count': 'Número de Municípios'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Box plots para cada feature por cluster
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

features_to_plot = [
    ('population_2022', 'População 2022', True),
    ('literacy_rate_2022', 'Taxa Alfabetização 2022 (%)', False),
    ('avg_income_2022', 'Renda Média 2022 (R$)', False),
    ('households_2022', 'Domicílios 2022', True),
    ('population_change_pct', 'Var. População (%)', False),
    ('literacy_change_pp', 'Var. Alfabetização (pp)', False),
    ('income_change_pct', 'Var. Renda (%)', False),
    ('households_change_pct', 'Var. Domicílios (%)', False)
]

for ax, (col, title, use_log) in zip(axes, features_to_plot):
    data = np.log10(df[col]) if use_log else df[col]
    ylabel = f'Log10({col})' if use_log else col
    df.boxplot(column=col if not use_log else None, by='cluster', ax=ax)
    if use_log:
        for i, cluster in enumerate(sorted(df['cluster'].unique())):
            cluster_data = np.log10(df[df['cluster'] == cluster][col])
            ax.boxplot(cluster_data, positions=[i+1])
    ax.set_title(title)
    ax.set_xlabel('Cluster')

plt.suptitle('Distribuições das Features por Cluster', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico radar para perfis dos clusters
# Normalizar médias dos clusters para comparação
cluster_means = df.groupby('cluster')[raw_features].mean()
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

# Selecionar features principais para o radar
radar_features = ['population_2022', 'literacy_rate_2022', 'avg_income_2022', 
                  'households_2022', 'population_change_pct', 'income_change_pct']

fig = go.Figure()

for cluster in sorted(df['cluster'].unique()):
    values = cluster_means_norm.loc[cluster, radar_features].values.tolist()
    values.append(values[0])  # Fechar o radar
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_features + [radar_features[0]],
        fill='toself',
        name=f'Cluster {cluster}'
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1])
    ),
    title='Perfis dos Clusters (Normalizados)',
    height=600
)
fig.show()

### 4.4 Interpretação dos Clusters

In [ ]:
# Gerar interpretações dos clusters baseado nas estatísticas
print("=" * 80)
print("INTERPRETAÇÃO DOS CLUSTERS")
print("=" * 80)

for cluster in sorted(df['cluster'].unique()):
    cluster_data = df[df['cluster'] == cluster]
    n_muni = len(cluster_data)
    
    print(f"\n--- CLUSTER {cluster} ({n_muni} municípios, {100*n_muni/len(df):.1f}%) ---")
    
    # População
    pop_mean = cluster_data['population_2022'].mean()
    pop_median = cluster_data['population_2022'].median()
    pop_size = "Grande" if pop_mean > df['population_2022'].mean() else "Pequena"
    print(f"  População: {pop_size} (média={pop_mean:,.0f}, mediana={pop_median:,.0f})")
    
    # Alfabetização
    lit_mean = cluster_data['literacy_rate_2022'].mean()
    lit_level = "Alta" if lit_mean > df['literacy_rate_2022'].mean() else "Baixa"
    print(f"  Alfabetização: {lit_level} ({lit_mean:.1f}%)")
    
    # Renda
    inc_mean = cluster_data['avg_income_2022'].mean()
    inc_level = "Alta" if inc_mean > df['avg_income_2022'].mean() else "Baixa"
    print(f"  Renda: {inc_level} (R$ {inc_mean:,.2f})")
    
    # Crescimento
    pop_change = cluster_data['population_change_pct'].mean()
    growth = "Crescendo" if pop_change > 0 else "Declinando"
    print(f"  Tendência Populacional: {growth} ({pop_change:+.1f}%)")
    
    # Principais regiões
    top_regions = cluster_data['region_name'].value_counts().head(2)
    print(f"  Principais Regiões: {', '.join(top_regions.index)}")

In [ ]:
# Exemplos de municípios de cada cluster
print("\n" + "=" * 80)
print("EXEMPLOS DE MUNICÍPIOS DE CADA CLUSTER")
print("=" * 80)

for cluster in sorted(df['cluster'].unique()):
    print(f"\n--- Cluster {cluster} ---")
    examples = df[df['cluster'] == cluster].nlargest(5, 'population_2022')[
        ['municipality_name', 'state_name', 'population_2022', 'avg_income_2022', 'literacy_rate_2022']
    ]
    print(examples.to_string(index=False))

## 5. Resumo e Conclusões

In [ ]:
print("=" * 80)
print("RESUMO DA ANÁLISE")
print("=" * 80)

print(f"""
DATASET:
  - Total de municípios analisados: {len(df):,}
  - Features utilizadas: {len(raw_features)}
  - Sem valores ausentes ou duplicatas

RESULTADOS DO PCA:
  - Componentes necessários para 80% da variância: {n_components_80}
  - Componentes necessários para 90% da variância: {n_components_90}
  - Primeiros 3 componentes explicam: {pca_3.explained_variance_ratio_.sum()*100:.1f}% da variância

AGRUPAMENTO K-MEANS:
  - K ótimo: {OPTIMAL_K} clusters
  - Coeficiente de Silhueta: {silhouette_score(X_cluster, df['cluster']):.4f}
  - Tamanhos dos clusters: {dict(df['cluster'].value_counts().sort_index())}

PRINCIPAIS ACHADOS:
  - Municípios podem ser agrupados baseado em características socioeconômicas
  - Tamanho da população e renda são fatores diferenciadores importantes
  - Padrões regionais são visíveis na composição dos clusters
  - Trajetórias de crescimento (2010-2022) variam significativamente entre clusters
""")

In [ ]:
# Salvar dados com clusters
output_columns = ['municipality_code', 'municipality_name', 'state_code', 'state_name',
                  'region_code', 'region_name', 'cluster', 'PC1', 'PC2', 'PC3'] + raw_features

df_output = df[output_columns].copy()
print(f"Dataset de saída pronto com {len(df_output)} linhas e {len(output_columns)} colunas")
df_output.head()

In [ ]:
# Salvar no S3 (opcional)
# output_key = 'gold/clustered_municipalities/data.parquet'
# with tempfile.NamedTemporaryFile(suffix='.parquet') as tmp:
#     df_output.to_parquet(tmp.name, index=False)
#     aws_profile = os.getenv("AWS_PROFILE", "mba-thesis")
#     session = boto3.Session(profile_name=aws_profile)
#     s3 = session.client('s3')
#     s3.upload_file(tmp.name, BUCKET_NAME, output_key)
#     print(f"Salvo em s3://{BUCKET_NAME}/{output_key}")

---

## Fim da Análise

Este notebook demonstrou:
1. Carregamento e validação do dataset consolidado de municípios
2. Estatísticas descritivas e distribuições de features
3. PCA para redução de dimensionalidade (identificando componentes de variância principais)
4. Agrupamento K-means para agrupar municípios por características socioeconômicas
5. Perfil e interpretação dos clusters

**Próximos Passos:**
- Usar rótulos de clusters para análise estratificada
- Investigar padrões de compliance dentro de cada cluster
- Comparar características dos clusters com dados de sanções